In [1]:
"""
ChromaDB

Chroma is a free-to-use lightweight vector DB that we can use to store our
various documents, metadata and their vector embeddings

Recall from yesterday we started making much higher level embeddings using the
sentence transformer, it converts it to a 384-dimensional vector.
"""

%pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 109.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.8 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found

In [2]:
# Imports
import os
import shutil
import pandas as pd
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer

In [4]:
# Let's create a small product catalog

products = [
    {
        "id": "P001",
        "title": "Trail Runner Pro Shoes",
        "category": "fitness",
        "price_tier": "premium",
        "tags": ["running", "trail", "shoes", "outdoor"],
        "description": "Lightweight trail running shoes with grip, cushioning, and breathable mesh for outdoor runs."
    },
    {
        "id": "P002",
        "title": "Summit Waterproof Hiking Boots",
        "category": "outdoor",
        "price_tier": "premium",
        "tags": ["hiking", "boots", "waterproof", "trail"],
        "description": "Durable waterproof hiking boots with ankle support for rocky trails and long outdoor hikes."
    },
    {
        "id": "P003",
        "title": "Performance Compression Socks",
        "category": "fitness",
        "price_tier": "budget",
        "tags": ["running", "recovery", "socks", "training"],
        "description": "Compression socks designed for running, lifting, recovery, and long training sessions."
    },
    {
        "id": "P004",
        "title": "Hydration Running Backpack",
        "category": "outdoor",
        "price_tier": "mid",
        "tags": ["running", "hydration", "backpack", "trail"],
        "description": "Compact hydration backpack for trail running, hiking, and endurance workouts."
    },
    {
        "id": "P005",
        "title": "Adjustable Dumbbell Set",
        "category": "fitness",
        "price_tier": "premium",
        "tags": ["strength", "weights", "home gym", "lifting"],
        "description": "Adjustable dumbbells for strength training, home workouts, and progressive overload."
    },
    {
        "id": "P006",
        "title": "Resistance Band Kit",
        "category": "fitness",
        "price_tier": "budget",
        "tags": ["strength", "mobility", "bands", "home gym"],
        "description": "Portable resistance bands for strength training, warmups, rehab, and mobility work."
    },
    {
        "id": "P007",
        "title": "Extra Thick Yoga Mat",
        "category": "fitness",
        "price_tier": "mid",
        "tags": ["yoga", "mobility", "stretching", "floor"],
        "description": "Cushioned yoga mat for stretching, mobility training, pilates, and floor exercises."
    },
    {
        "id": "P008",
        "title": "Deep Tissue Foam Roller",
        "category": "fitness",
        "price_tier": "budget",
        "tags": ["recovery", "mobility", "massage", "training"],
        "description": "Textured foam roller for muscle recovery, mobility, warmups, and post-workout soreness."
    },
    {
        "id": "P009",
        "title": "GPS Running Smartwatch",
        "category": "electronics",
        "price_tier": "premium",
        "tags": ["running", "watch", "gps", "fitness"],
        "description": "Fitness smartwatch with GPS tracking, heart-rate monitoring, workout metrics, and route tracking."
    },
    {
        "id": "P010",
        "title": "Sweatproof Wireless Earbuds",
        "category": "electronics",
        "price_tier": "mid",
        "tags": ["audio", "running", "gym", "wireless"],
        "description": "Wireless earbuds with sweat resistance and secure fit for workouts, running, and commuting."
    },
    {
        "id": "P011",
        "title": "Burr Coffee Grinder",
        "category": "kitchen",
        "price_tier": "mid",
        "tags": ["coffee", "grinder", "espresso", "kitchen"],
        "description": "Consistent burr grinder for espresso, pour-over, drip coffee, and fresh coffee preparation."
    },
    {
        "id": "P012",
        "title": "Compact Espresso Machine",
        "category": "kitchen",
        "price_tier": "premium",
        "tags": ["coffee", "espresso", "kitchen", "morning"],
        "description": "Compact espresso machine for making lattes, cappuccinos, and café-style coffee at home."
    },
    {
        "id": "P013",
        "title": "Cold Brew Coffee Maker",
        "category": "kitchen",
        "price_tier": "budget",
        "tags": ["coffee", "cold brew", "kitchen", "drink"],
        "description": "Simple cold brew maker for smooth iced coffee and easy overnight brewing."
    },
    {
        "id": "P014",
        "title": "Cast Iron Skillet",
        "category": "kitchen",
        "price_tier": "mid",
        "tags": ["cooking", "skillet", "kitchen", "durable"],
        "description": "Heavy-duty cast iron skillet for searing, baking, frying, and high-heat cooking."
    },
    {
        "id": "P015",
        "title": "Digital Air Fryer",
        "category": "kitchen",
        "price_tier": "mid",
        "tags": ["cooking", "air fryer", "meal prep", "kitchen"],
        "description": "Digital air fryer for quick meals, crispy vegetables, reheating leftovers, and meal prep."
    },
    {
        "id": "P016",
        "title": "Noise-Canceling Headphones",
        "category": "electronics",
        "price_tier": "premium",
        "tags": ["audio", "headphones", "work", "travel"],
        "description": "Noise-canceling headphones for focus, travel, remote work, and high-quality audio."
    },
    {
        "id": "P017",
        "title": "Ergonomic Mechanical Keyboard",
        "category": "electronics",
        "price_tier": "premium",
        "tags": ["keyboard", "work", "desk", "typing"],
        "description": "Mechanical keyboard with ergonomic layout for coding, writing, and long work sessions."
    },
    {
        "id": "P018",
        "title": "Adjustable Laptop Stand",
        "category": "office",
        "price_tier": "budget",
        "tags": ["desk", "work", "ergonomic", "laptop"],
        "description": "Adjustable laptop stand for better posture, remote work, and ergonomic desk setups."
    },
    {
        "id": "P019",
        "title": "Portable Second Monitor",
        "category": "electronics",
        "price_tier": "mid",
        "tags": ["monitor", "work", "travel", "productivity"],
        "description": "Slim portable monitor for productivity, travel workstations, coding, and presentations."
    },
    {
        "id": "P020",
        "title": "Smart Desk Lamp",
        "category": "office",
        "price_tier": "mid",
        "tags": ["desk", "lighting", "work", "smart"],
        "description": "Smart adjustable desk lamp with brightness controls for studying, reading, and remote work."
    },
    {
        "id": "P021",
        "title": "Lightweight Rain Jacket",
        "category": "outdoor",
        "price_tier": "mid",
        "tags": ["rain", "jacket", "hiking", "travel"],
        "description": "Packable rain jacket for hiking, travel, commuting, and unpredictable weather."
    },
    {
        "id": "P022",
        "title": "Insulated Water Bottle",
        "category": "outdoor",
        "price_tier": "budget",
        "tags": ["hydration", "water", "hiking", "gym"],
        "description": "Insulated water bottle that keeps drinks cold during workouts, hiking, commuting, and travel."
    },
    {
        "id": "P023",
        "title": "Travel Packing Cubes",
        "category": "travel",
        "price_tier": "budget",
        "tags": ["travel", "organization", "packing", "luggage"],
        "description": "Packing cubes for organizing clothes, luggage, weekend trips, and long travel days."
    },
    {
        "id": "P024",
        "title": "High-Capacity Portable Charger",
        "category": "electronics",
        "price_tier": "mid",
        "tags": ["battery", "travel", "phone", "charger"],
        "description": "Portable charger for phones, tablets, travel, commuting, and long days away from outlets."
    },
    {
        "id": "P025",
        "title": "Camping Hammock",
        "category": "outdoor",
        "price_tier": "budget",
        "tags": ["camping", "hiking", "outdoor", "relaxing"],
        "description": "Lightweight camping hammock for backpacking, hiking breaks, campsites, and outdoor relaxation."
    },
    {
        "id": "P026",
        "title": "Reflective Dog Leash",
        "category": "pet",
        "price_tier": "budget",
        "tags": ["dog", "walking", "reflective", "safety"],
        "description": "Durable reflective dog leash for safe evening walks, training, and daily pet care."
    },
    {
        "id": "P027",
        "title": "Interactive Dog Puzzle Toy",
        "category": "pet",
        "price_tier": "budget",
        "tags": ["dog", "toy", "training", "enrichment"],
        "description": "Interactive dog puzzle toy for mental stimulation, treat training, and indoor enrichment."
    },
    {
        "id": "P028",
        "title": "Protein Shaker Bottle",
        "category": "fitness",
        "price_tier": "budget",
        "tags": ["protein", "gym", "nutrition", "workout"],
        "description": "Leakproof shaker bottle for protein shakes, pre-workout drinks, gym bags, and meal prep."
    },
    {
        "id": "P029",
        "title": "Meal Prep Container Set",
        "category": "kitchen",
        "price_tier": "budget",
        "tags": ["meal prep", "nutrition", "storage", "kitchen"],
        "description": "Reusable meal prep containers for portion control, leftovers, lunches, and weekly planning."
    },
    {
        "id": "P030",
        "title": "Compact Rowing Machine",
        "category": "fitness",
        "price_tier": "premium",
        "tags": ["cardio", "home gym", "rowing", "fitness"],
        "description": "Compact rowing machine for low-impact cardio, endurance training, and home gym workouts."
    },
]

df = pd.DataFrame(products)

df.head()

,id,title,category,price_tier,tags,description
0,P001,Trail Runner Pro Shoes,fitness,premium,"[running, trail, shoes, outdoor]","Lightweight trail running shoes with grip, cus..."
1,P002,Summit Waterproof Hiking Boots,outdoor,premium,"[hiking, boots, waterproof, trail]",Durable waterproof hiking boots with ankle sup...
2,P003,Performance Compression Socks,fitness,budget,"[running, recovery, socks, training]","Compression socks designed for running, liftin..."
3,P004,Hydration Running Backpack,outdoor,mid,"[running, hydration, backpack, trail]","Compact hydration backpack for trail running, ..."
4,P005,Adjustable Dumbbell Set,fitness,premium,"[strength, weights, home gym, lifting]","Adjustable dumbbells for strength training, ho..."


In [5]:
"""
Let's create our sentence transformer and create the text to embed
"""

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Collect our text to embed/encode
df["search_text"] = (
    df["title"] + ". "
    + "Category: " + df["category"] + ". "
    + "Tags: " + df["tags"].apply(lambda tags: ", ".join(tags)) + ". "
    + df["description"]
)

df.head()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

,id,title,category,price_tier,tags,description,search_text
0,P001,Trail Runner Pro Shoes,fitness,premium,"[running, trail, shoes, outdoor]","Lightweight trail running shoes with grip, cus...",Trail Runner Pro Shoes. Category: fitness. Tag...
1,P002,Summit Waterproof Hiking Boots,outdoor,premium,"[hiking, boots, waterproof, trail]",Durable waterproof hiking boots with ankle sup...,Summit Waterproof Hiking Boots. Category: outd...
2,P003,Performance Compression Socks,fitness,budget,"[running, recovery, socks, training]","Compression socks designed for running, liftin...",Performance Compression Socks. Category: fitne...
3,P004,Hydration Running Backpack,outdoor,mid,"[running, hydration, backpack, trail]","Compact hydration backpack for trail running, ...",Hydration Running Backpack. Category: outdoor....
4,P005,Adjustable Dumbbell Set,fitness,premium,"[strength, weights, home gym, lifting]","Adjustable dumbbells for strength training, ho...",Adjustable Dumbbell Set. Category: fitness. Ta...


In [7]:
# Actually Embed our search text

texts = df["search_text"].tolist()

embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
    # Batch size controls how many records get processed at once
    batch_size=16 # Processes 16 documents in parallel
)

print("Number of Products: ", len(embeddings))
print("Number of Embedding Dimensions: ", len(embeddings[0]))

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Number of Products:  30
Number of Embedding Dimensions:  384


In [8]:
"""
Let's set up ChromaDB and create our first collection

Vector DBs typically store collections of documents including:
-id
-document text
-embeddings
-metadata

We'll go ahead and create a ChromaDB instance that will store in a file, so we can pull
it out later if we want
"""

DB_PATH = "./chroma_product_recs"
COLLECTION_NAME = "product_recommendations"

# Clean previous data if required
if os.path.exists(DB_PATH):
  shutil.rmtree(DB_PATH)

# Just like yesterday we need some sort of connection to our DB
# In Chroma we use a client

client = chromadb.PersistentClient(path=DB_PATH)

# Create a collection (basically a table of documents)
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    # METADATA ABOUT THE COLLECTION NOT ABOUT A SPECIFIC DOCUMENT
    metadata = {"hnsw:space" : "cosine"} # Use cosine similarity to calculate distance and then store in HNSW
)

"""
What on Earth is HNSW?
"""
print("Collection Ready")

Collection Ready


In [12]:
"""
VectorDBs are just like regular databases and they have to be able to manage
data.

Commands
Add -> Creates a New records (errors on dupes)
Update -> Modifies an existing records (error on a missing record)
Get -> Allows us to get a record based off id
Delete -> Allows us to delete a record

Upsert -> Combo of Insert and Update -> Inserts record if it doesn't exist and updates
record if it does exist
Query -> Think of this an a vector similarity search, used for querying the DB with
an embedded vector
"""

metadatas = []

# We need to construct the metadata for each row in the df since it has
# multiple values (category, price tier, title and tags)
# Why do we use metadata? It's used for filtering records later on

for _, row in df.iterrows():
  metadatas.append({
      "title": row["title"],
      "category": row["category"],
      "price_tier": row["price_tier"],
      # Chroma likes simple values in its metadata so combine tags again
      "tags": ", ".join(row["tags"])
  })

  # Group Upsert
  # Add in all of our records at once
collection.upsert(
    ids=df["id"].tolist(),
    documents=df["search_text"].tolist(),
    embeddings=[embedding.tolist() for embedding in embeddings],
    metadatas=metadatas
)

  # Verify they've been added
print("Products in Collection: ", collection.count())

Products in Collection:  30


In [13]:
def results_to_df(results):
    rows = []

    ids = results["ids"][0]
    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]

    for product_id, document, metadata, distance in zip(ids, documents, metadatas, distances):
        rows.append({
            "id": product_id,
            "title": metadata["title"],
            "category": metadata["category"],
            "price_tier": metadata["price_tier"],
            "tags": metadata["tags"],
            "distance": round(distance, 4),
            # For cosine distance in Chroma, similarity is roughly 1 - distance.
            "approx_similarity": round(1 - distance, 4)
        })

    return pd.DataFrame(rows)

def semantic_product_search(query, top_k=5, where=None):
  # Recall yesterday the steps for semantic searching
  # We embed our query and then search our db for similar vectors
  # based on the cosine similary

  # REMINDER: Use the same model to embed your documents and your queries
  query_embedding = model.encode(
      [query],
      normalize_embeddings=True
  )[0].tolist()

  # Query our collection
  # This is where Chroma abstracts some of the logic we did yesterday
  results = collection.query(
      query_embeddings = [query_embedding],
      n_results = top_k,
      where=where, # Used for metadata filters
      # This defines what results we want
      include = ["documents", "metadatas", "distances"]
  )

  return results_to_df(results)

In [14]:
# Try some semantic product searching
semantic_product_search("I need gear for long trails runs and hiking")

,id,title,category,price_tier,tags,distance,approx_similarity
0,P001,Trail Runner Pro Shoes,fitness,premium,"running, trail, shoes, outdoor",0.5579,0.4421
1,P002,Summit Waterproof Hiking Boots,outdoor,premium,"hiking, boots, waterproof, trail",0.5693,0.4307
2,P004,Hydration Running Backpack,outdoor,mid,"running, hydration, backpack, trail",0.5998,0.4002
3,P006,Resistance Band Kit,fitness,budget,"strength, mobility, bands, home gym",0.6520,0.3480
4,P021,Lightweight Rain Jacket,outdoor,mid,"rain, jacket, hiking, travel",0.6852,0.3148


In [20]:
# Searching with a metadata filter
# Using this basically shrinks the number of vectors we're searching through
semantic_product_search(
    "something for strength training at home",
    top_k=3,
    where = {"category": "fitness"}
)

,id,title,category,price_tier,tags,distance,approx_similarity
0,P005,Adjustable Dumbbell Set,fitness,premium,"strength, weights, home gym, lifting",0.5307,0.4693
1,P006,Resistance Band Kit,fitness,budget,"strength, mobility, bands, home gym",0.5834,0.4166
2,P030,Compact Rowing Machine,fitness,premium,"cardio, home gym, rowing, fitness",0.6499,0.3501


In [21]:
"""
Given a product how do I find related products

Compare the existing embedding I already have for the document or product to
all of the other ones

This is called item-to-item recommendation
"""

def get_product(product_id):
  product = df[df["id"] == product_id]

  if product.empty:
    raise ValueError("Unknown product ID")

  return product.iloc[0]
  # Return the value for the single product as a DF

def recommend_similar_products(product_id, top_k=5, same_category_only=False):
  # Get access to product info
  product = get_product(product_id)

  # Fetch the existing embedding for our product from Chroma
  # This is different than Query because I have an ID value I'm trying to pull
  # an exact document

  stored = collection.get(
      ids=[product_id],
      include=["embeddings", "documents", "metadatas"]
  )

  product_embedding = stored["embeddings"][0]

  where = None
  if same_category_only:
    # If we only want the same category, filter on the category itself
    where = {"category": product["category"]}

  # Since we need to pull other documents based of similarity, we QUERY
  results = collection.query(
      query_embeddings = [product_embedding],
      # When we query based off a product, the product itself will likely return
      # as the most similar vector, so we add 1 to k to account for this and we'll
      # filter it out later
      n_results = top_k + 1,
      where = where,
      include=["documents", "metadatas", "distances"]
  )

  recs = results_to_df(results)

  # Remove the initial product if it is there
  recs = recs[recs["id"] != product_id].head(top_k).reset_index(drop=True)

  print(f"Because you viewed: {product["title"]}")
  return recs

In [22]:
recommend_similar_products("P001")

Because you viewed: Trail Runner Pro Shoes


,id,title,category,price_tier,tags,distance,approx_similarity
0,P004,Hydration Running Backpack,outdoor,mid,"running, hydration, backpack, trail",0.3604,0.6396
1,P003,Performance Compression Socks,fitness,budget,"running, recovery, socks, training",0.3653,0.6347
2,P002,Summit Waterproof Hiking Boots,outdoor,premium,"hiking, boots, waterproof, trail",0.3798,0.6202
3,P009,GPS Running Smartwatch,electronics,premium,"running, watch, gps, fitness",0.5128,0.4872
4,P010,Sweatproof Wireless Earbuds,electronics,mid,"audio, running, gym, wireless",0.5129,0.4871


In [23]:
recommend_similar_products("P001", same_category_only=True)

Because you viewed: Trail Runner Pro Shoes


,id,title,category,price_tier,tags,distance,approx_similarity
0,P003,Performance Compression Socks,fitness,budget,"running, recovery, socks, training",0.3653,0.6347
1,P006,Resistance Band Kit,fitness,budget,"strength, mobility, bands, home gym",0.5560,0.4440
2,P007,Extra Thick Yoga Mat,fitness,mid,"yoga, mobility, stretching, floor",0.5847,0.4153
3,P028,Protein Shaker Bottle,fitness,budget,"protein, gym, nutrition, workout",0.6048,0.3952
4,P030,Compact Rowing Machine,fitness,premium,"cardio, home gym, rowing, fitness",0.6221,0.3779


In [30]:
"""
User Profile Recommendations

What we're going to do is take the list of products the User has liked and the
products they have already bought. We're going to combine the vectors and
search for a similar product(s) to recommend

You've liked these and bought this, check out this recommend product

Sometimes referred to as a User Profile Vector
"""

def build_user_profile_embedding(liked_product_ids):
  # Get the existing embeddings from Chroma
  stored = collection.get(
      ids=liked_product_ids,
      include=["embeddings"]
  )

  # We'll create a Numpy Array of the embeddings and take the average of them
  vectors = np.array(stored["embeddings"])

  # Average all of them
  profile_vector = vectors.mean(axis=0)

  # Renormalize the vector to keep it aligned with our other inputs
  profile_vector = profile_vector / np.linalg.norm(profile_vector)

  return profile_vector.tolist()

def recommend_for_user(liked_product_ids, already_owned_ids=None, top_k=5, where=None):
  # If the user does not own any products let's initialize a blank list
  if already_owned_ids is None:
    already_owned_ids = []

  # Let's create a set of items to exclude from the recommendations
  # Remove the items that we have already bought and already liked
  # Sets remove duplicates
  exclude_ids = list(set(liked_product_ids + already_owned_ids))

  # Create an embedding for the user
  profile_embedding = build_user_profile_embedding(exclude_ids)

  # Now that we have an embedding we can do a similarity search and get our
  # products
  results = collection.query(
      query_embeddings=[profile_embedding],
      n_results = top_k + len(exclude_ids),
      where=where,
      include=["documents", "metadatas", "distances"]
  )

  # Convert the results to a dataframe and filter out results we already have
  recs = results_to_df(results)
  recs = recs[~recs["id"].isin(exclude_ids)].head(top_k).reset_index(drop=True)

  return recs

In [31]:
# Example user
# They like running hiking, hydration and fitness tech
liked_ids = ["P001", "P004", "P009"]
already_owned = ["P022"]

recommend_for_user(
    liked_product_ids=liked_ids,
    already_owned_ids = already_owned
)

,id,title,category,price_tier,tags,distance,approx_similarity
0,P002,Summit Waterproof Hiking Boots,outdoor,premium,"hiking, boots, waterproof, trail",0.3223,0.6777
1,P025,Camping Hammock,outdoor,budget,"camping, hiking, outdoor, relaxing",0.3779,0.6221
2,P010,Sweatproof Wireless Earbuds,electronics,mid,"audio, running, gym, wireless",0.3789,0.6211
3,P003,Performance Compression Socks,fitness,budget,"running, recovery, socks, training",0.3895,0.6105
4,P028,Protein Shaker Bottle,fitness,budget,"protein, gym, nutrition, workout",0.4207,0.5793


In [32]:
# Example user
# They like office stuff
liked_ids = ["P018", "P019"]

recommend_for_user(
    liked_product_ids=liked_ids,
)

,id,title,category,price_tier,tags,distance,approx_similarity
0,P017,Ergonomic Mechanical Keyboard,electronics,premium,"keyboard, work, desk, typing",0.3311,0.6689
1,P020,Smart Desk Lamp,office,mid,"desk, lighting, work, smart",0.3567,0.6433
2,P024,High-Capacity Portable Charger,electronics,mid,"battery, travel, phone, charger",0.4320,0.5680
3,P009,GPS Running Smartwatch,electronics,premium,"running, watch, gps, fitness",0.5142,0.4858
4,P016,Noise-Canceling Headphones,electronics,premium,"audio, headphones, work, travel",0.5957,0.4043


In [ ]:
"""
Metrics
Precision@K
Recall@K
Lots of others!

Precision@K for K = 5
How many of the 5 returned records are relevant  / K
4 relevant recommendations / 5 -> Precision@5 is .8 or 80%